# Text Summarization & ROUGE Evaluation

**Refurbished for 🤗 Transformers v5 (2026).**

This notebook does the same two things as the original demo:

1. **Summarize** text with a few Hugging Face models
   (`bart-large-cnn`, `Falconsai/text_summarization`, `distilbart-cnn-12-6`).
2. **Judge** the summaries against reference summaries using **ROUGE**, then
   surface the best and worst ones.

### What changed and why

In **Transformers v5**
the `summarization` pipeline task (along with `translation`, `text2text-generation`
and `question-answering`) was **removed**. The *models* are unchanged and fully
supported — only the pipeline wrapper is gone. So we load each seq2seq model
directly with `AutoModelForSeq2SeqLM` + `.generate()`, wrapped in a tiny
`Summarizer` helper that returns the same `[{"summary_text": ...}]` shape the old
pipeline did.

The removal of task-specific pipelines like summarization, translation, and text2text-generation in Transformers v5 was part of a major architectural overhaul. Hugging Face aimed to eliminate nearly half a decade of accumulated technical debt and streamline the library around modern generative interfaces.

Modern NLP treats almost everything as a unified text generation or instruction-following problem. The "task" is now defined by your prompt, system instructions, or model weights, rather than the Python class wrapping the code.

### Runtime

- Pick a **GPU** runtime (Runtime → Change runtime type → GPU). The helper falls
  back to CPU automatically if no GPU is present, just slower.

### Libraries

**`rouge_score`** — computes the ROUGE metric (Recall-Oriented Understudy for Gisting
Evaluation), the standard for scoring summary quality against references.

In [1]:
!pip install -q -U datasets rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.7 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Using device:", "cuda" if torch.cuda.is_available() else "cpu")

Using device: cuda


### A drop-in replacement for `pipeline("summarization", ...)`

`Summarizer` loads the seq2seq model + tokenizer directly and calls `.generate()`.
Calling the instance returns `[{"summary_text": ...}]`, exactly like the old
summarization pipeline, so the rest of the notebook stays familiar.

Notes:

- Inputs are truncated to the model's max input length (BART/DistilBART = 1024).

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


class Summarizer:
    """Simple replacement for pipeline("summarization", ...)."""

    def __init__(self, model_name, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def __call__(self, texts):
        texts = [texts] if isinstance(texts, str) else list(texts)

        inputs = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(self.device)

        with torch.no_grad():
            output_ids = self.model.generate(**inputs)

        summaries = self.tokenizer.batch_decode(
            output_ids,
            skip_special_tokens=True
        )

        return [{"summary_text": summary.strip()} for summary in summaries]

In [4]:
content = """
  We are on the brink of a technological revolution that could jumpstart productivity, boost global growth and raise incomes around the world.
  Yet it could also replace jobs and deepen inequality. The rapid advance of artificial intelligence has captivated the world, causing both excitement and alarm, and
  raising important questions about its potential impact on the global economy. The net effect is difficult to foresee, as AI will ripple through economies in complex ways.
  What we can say with some confidence is that we will need to come up with a set of policies to safely leverage the vast potential of AI for the benefit of humanity.

  In a new analysis, IMF staff examine the potential impact of AI on the global labor market. Many studies have predicted the likelihood that jobs will be replaced by AI.
  Yet we know that in many cases AI is likely to complement human work. The IMF analysis captures both these forces.

  The findings are striking: almost 40 percent of global employment is exposed to AI. Historically, automation and information technology have tended to affect routine tasks,
  but one of the things that sets AI apart is its ability to impact high-skilled jobs. As a result, advanced economies face greater risks from AI-but also more opportunities to
  leverage its benefits-compared with emerging market and developing economies.

  In advanced economies, about 60 percent of jobs may be impacted by AI. Roughly half the exposed jobs may benefit from AI integration, enhancing productivity. For the other
  half, AI applications may execute key tasks currently performed by humans, which could lower labor demand, leading to lower wages and reduced hiring. In the most extreme
  cases, some of these jobs may disappear.

  In emerging markets and low-income countries, by contrast, AI exposure is expected to be 40 percent and 26 percent, respectively. These findings suggest emerging market and
  developing economies face fewer immediate disruptions from AI. At the same time, many of these countries don't have the infrastructure or skilled workforces to harness the
  benefits of AI, raising the risk that over time the technology could worsen inequality among nations.
"""

print(content)


  We are on the brink of a technological revolution that could jumpstart productivity, boost global growth and raise incomes around the world.
  Yet it could also replace jobs and deepen inequality. The rapid advance of artificial intelligence has captivated the world, causing both excitement and alarm, and
  raising important questions about its potential impact on the global economy. The net effect is difficult to foresee, as AI will ripple through economies in complex ways.
  What we can say with some confidence is that we will need to come up with a set of policies to safely leverage the vast potential of AI for the benefit of humanity.

  In a new analysis, IMF staff examine the potential impact of AI on the global labor market. Many studies have predicted the likelihood that jobs will be replaced by AI.
  Yet we know that in many cases AI is likely to complement human work. The IMF analysis captures both these forces.

  The findings are striking: almost 40 percent of global emp

### `facebook/bart-large-cnn`

On the Hub: Models → filter by **Summarization** → `facebook/bart-large-cnn` is a
top result. https://huggingface.co/facebook/bart-large-cnn

BART is an encoder–decoder (seq2seq) model, so it loads cleanly with
`AutoModelForSeq2SeqLM`.

In [5]:
summarizer = Summarizer("facebook/bart-large-cnn")

summarizer

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [6]:
summarizer.device

'cuda'

In [7]:
summarizer(content)

[{'summary_text': 'The rapid advance of artificial intelligence has captivated the world. IMF staff examine the potential impact of AI on the global labor market. In advanced economies, about 60 percent of jobs may be impacted by AI. In emerging markets and low-income countries, by contrast, AI exposure is expected to be 40 percent.'}]

### Load the news dataset

https://drive.google.com/drive/u/0/folders/11AKRIT_VEkg_f3XsIVNik6w-YB6FZj8Z

In [9]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import os
import itertools
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/oreilly_news_articles/news_articles.csv"

news_df = pd.read_csv(CSV_PATH)

news_df.sample(10)

,Content,Summary,Dataset
4605,Seven inmates were evacuated and the fire brig...,Prisoners ran riot after fire broke out at a n...,XSum
102,Roberto Martinez insists he is under no pressu...,Roberto Martinez insists he is under no pressu...,CNN/Daily Mail
3999,The data has reportedly been leaked on the so ...,"Customer data stolen from Ashley Madison, a da...",XSum
1846,Washington (CNN) -- Longtime White House repor...,NEW: Rabbi who recorded interview says Thomas ...,CNN/Daily Mail
1294,Hong Kong (CNN) -- How's this for a bright ide...,Blindingly simple idea sees Chinese police usi...,CNN/Daily Mail
2235,"Disgraced: Councillor Robert Bleakley, 43, sen...","Councillor Robert Bleakley, 43, racked up a £2...",CNN/Daily Mail
604,The company's Hong Kong-listed shares rose mor...,Shares in embattled mining giant Glencore have...,XSum
1873,"Unbowed: John Tulloch, 63, was a victim of the...",John Tulloch - born to British parents in colo...,CNN/Daily Mail
2757,As the former director of communications at th...,Costume designer Lyn Paolo puts 'Scandal's' Ol...,CNN/Daily Mail
2940,"Kim Davis, an elected official in Rowan County...",Hundreds have gathered in support of a county ...,XSum


In [11]:
news_df.shape

(5000, 3)

Null value check

In [12]:
news_df.isnull().sum()

,0
Content,0
Summary,0
Dataset,0


In [13]:
import re

def clean_txt(txt):
    txt = txt.lower()

    # Remove backslashes, @ symbols
    txt = txt.replace("\\", " ")
    txt = txt.replace("@", " ")

    # Replace non-breaking space (used to help not split words if text is wrapped)
    txt = txt.replace("\xa0", " ")

    txt = txt.replace("/", " ")
    txt = txt.replace("\n", " ")
    txt = txt.replace("'s", " ")
    txt = txt.replace('"', ' ')

    # Remove extra whitespace
    txt = re.sub(r'\s+', ' ', txt).strip()

    return txt

In [14]:
cleaned_df = news_df.copy()

cleaned_df['Content'] = cleaned_df['Content'].map(clean_txt)
cleaned_df['Summary'] = cleaned_df['Summary'].map(clean_txt)

cleaned_df.sample(min(10, len(cleaned_df)))

,Content,Summary,Dataset
3979,(cnn) -- a 16-year-old nevada boy sought in th...,"boy, 16, was near las vegas strip in open-air ...",CNN/Daily Mail
3953,fifa executive committee will be asked to vote...,fifa executive committee will meet next week i...,CNN/Daily Mail
3269,"countless books, paintings and films have atte...","reconstruction was made using more than 190,00...",CNN/Daily Mail
429,"by . anna edwards . published: . 06:45 est, 8 ...",former apprentice winner spotted at job centre...,CNN/Daily Mail
809,by . james nye for mailonline . the incredible...,stephanie rey and her daughter were watching l...,CNN/Daily Mail
3327,"human evolution was driven by short, rapid wav...",researchers made discovery after analysing sed...,CNN/Daily Mail
2340,the parents of a 21-year-old australian jihadi...,"zehra duman, from melbourne, has married extre...",CNN/Daily Mail
483,almost a third of married women say they still...,thirty-one per cent of married women said they...,CNN/Daily Mail
3994,(cnn) -- one of the most enduring mysteries of...,researchers compared dna of remains with that ...,CNN/Daily Mail
4036,"(cnn) -- for alex zanardi, losing both legs in...",italian racecar driver alex zanardi lost his l...,CNN/Daily Mail


Checking an instance of Content and Summary

In [15]:
news_df['Content'][15]

"Ashton Kutcher has entered the firestorm surrounding under-fire taxi-hiring app Uber and defended controversial comments made by an executive who\xa0suggested spending $1 million to dig up dirt on journalists who criticize the company. Kutcher, an investor in the app, took to Twitter on Wednesday to show his support for beleaguered VP Emil Michael and described Sarah Lacy, a female journalist who has been highly critical of the company, as 'shady'. 'What is so wrong about digging up dirt on shady journalist?' tweeted the celebrity tech entrepreneur who has invested in tech firms including Skype, Foursquare, Airbnb and Spotify through his\xa0venture capital firm A-Grade. Under-fire: Ashton Kutcher has entered the firestorm surrounding taxi-hiring app Uber and defended Senior VP Emil Michael who suggested spending $1 million to dig up dirt on journalists who criticize the company . Outburst: Kutcher, an investor in the app, took to Twitter on Wednesday to attack Sarah Lacy, the journali

`\xa0`, `@` are removed

In [16]:
cleaned_df['Content'][15]

"ashton kutcher has entered the firestorm surrounding under-fire taxi-hiring app uber and defended controversial comments made by an executive who suggested spending $1 million to dig up dirt on journalists who criticize the company. kutcher, an investor in the app, took to twitter on wednesday to show his support for beleaguered vp emil michael and described sarah lacy, a female journalist who has been highly critical of the company, as hady'. 'what is so wrong about digging up dirt on shady journalist?' tweeted the celebrity tech entrepreneur who has invested in tech firms including skype, foursquare, airbnb and spotify through his venture capital firm a-grade. under-fire: ashton kutcher has entered the firestorm surrounding taxi-hiring app uber and defended senior vp emil michael who suggested spending $1 million to dig up dirt on journalists who criticize the company . outburst: kutcher, an investor in the app, took to twitter on wednesday to attack sarah lacy, the journalist who h

In [17]:
news_df['Summary'][15]

"Uber investor Kutcher has tweeted his support for the under-fire app and accused critic Sarah Lacy of being 'shady'\nKutcher's comments follow the firestorm that\xa0erupted\xa0after senior VP Emil\xa0Michael\xa0suggested Uber should hire a $1 million team of researchers .\nThey 'would dig up dirt on journalists' personal lives and their families'\nMichael was reportedly speaking with specific reference to Lacy, an\xa0outspoken critic of the online taxi service .\nActor quickly apologized but was strongly criticized by Twitter users - many of whom accused him of only getting involved because he is an investor .\nIn 2011 he was forced into an embarrassing climbdown after tweeting that the sacking of Joe Paterno as head coach of Penn State showed 'no class'"

New line characters are removed

In [18]:
cleaned_df['Summary'][15]

"uber investor kutcher has tweeted his support for the under-fire app and accused critic sarah lacy of being hady' kutcher comments follow the firestorm that erupted after senior vp emil michael suggested uber should hire a $1 million team of researchers . they 'would dig up dirt on journalists' personal lives and their families' michael was reportedly speaking with specific reference to lacy, an outspoken critic of the online taxi service . actor quickly apologized but was strongly criticized by twitter users - many of whom accused him of only getting involved because he is an investor . in 2011 he was forced into an embarrassing climbdown after tweeting that the sacking of joe paterno as head coach of penn state showed 'no class'"

### `sshleifer/distilbart-cnn-12-6`

https://huggingface.co/sshleifer/distilbart-cnn-12-6

#### BART (Bidirectional and Auto-Regressive Transformers)
BART is an encoder–decoder model from Facebook AI. The encoder reads the input
bidirectionally (BERT-like) and the decoder generates the output autoregressively
(GPT-like), which makes it strong for sequence generation tasks like summarization,
translation, and text generation.

#### DistilBART
DistilBART is a distilled (smaller, faster) version of BART: a compact "student" model
trained to mimic a larger "teacher," keeping most of the quality while using far fewer
resources — handy for limited-hardware or real-time use.

Docs for the generation API:
https://huggingface.co/docs/transformers/main/en/main_classes/text_generation

In [19]:
summarizer = Summarizer("sshleifer/distilbart-cnn-12-6")

summarizer

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

In [20]:
example_text = cleaned_df['Content'][8]

example_text

"many people might associate roses with romance novels, but in fact the smell of chocolate tempts readers to buy loved-up fiction. researchers at antwerp university in belgium found that while a chocolatey smell makes people hungry to buy romance novels, it does not necessarily tempt people to buy grittier genres such as crime or business books. the scientists, who permeated a bookshop with the perfume of chocolate, also found that the scent encouraged customers to browse through titles. the government funded study found that bookshop visitors were almost six times more likely to buy a romance novel if they smelt chocolate - and the same results applied to cookery books too . they studied the behaviour of 201 customers in a popular chain bookshop over ten days to find that customers were 3.5 times more likely to pick up romance novels when they smelt chocolate, the guardian reported. the government funded study also found that bookshop visitors were almost six times more likely to buy 

In [21]:
summary_txt = summarizer(example_text)

summary_txt

[{'summary_text': 'Researchers at antwerp university in belgium, Belgium, permeated a bookshop with the perfume of chocolate . They found customers were 3.5 times more likely to buy romance novels when they smelt chocolate . The government funded study also found the scent encouraged customers to browse through titles .'}]

In [22]:
summary_txt = clean_txt(summary_txt[0]['summary_text'])

summary_txt

'researchers at antwerp university in belgium, belgium, permeated a bookshop with the perfume of chocolate . they found customers were 3.5 times more likely to buy romance novels when they smelt chocolate . the government funded study also found the scent encouraged customers to browse through titles .'

In [23]:
ref_txt = cleaned_df['Summary'][8]

ref_txt

'belgium scientists flooded a bookshop with a chocolatey smell to find customers were almost six times more likely to buy a romance novel . the government-funded study said the scent does not have a big effect on making people want to purchase travel, crime, business or comic books . researchers at antwerp university said the sweet smell encouraged customers to browse and . sales of all books rose during the experiment .'

### Scoring summaries with ROUGE

ROUGE compares a generated summary to one or more reference summaries.

- **ROUGE-N** — overlap of n-grams. **ROUGE-1** = unigrams (individual word content),
  **ROUGE-2** = bigrams (captures word-order to a degree).
- **ROUGE-L** — longest common subsequence (LCS): shared words in order, not necessarily
  contiguous, so it's more forgiving of phrasing differences.
- **ROUGE-Lsum** — the summary-level variant of ROUGE-L that accounts for sentence
  structure; the standard headline metric for summarization.

Higher = closer to the reference.

We compute these with the `rouge_score` library directly (what 🤗 `evaluate` wraps).
The `compute_rouge` helper below mirrors `evaluate.load("rouge").compute(...)`:
`use_aggregator=True` returns mean f-measures, `False` returns per-example lists.

In [24]:
import re
import numpy as np
from rouge_score import rouge_scorer

ROUGE_TYPES = ["rouge1", "rouge2", "rougeL", "rougeLsum"]


def prep_for_rouge_lsum(text):
    sentences = re.split(r"(?<=[.!?])\s+|\n+", text.strip())
    return "\n".join(sentence for sentence in sentences if sentence)


def compute_rouge(predictions, references, use_aggregator=True):
    scorer = rouge_scorer.RougeScorer(
        ROUGE_TYPES,
        use_stemmer=True
    )

    scores = {rouge_type: [] for rouge_type in ROUGE_TYPES}

    for prediction, reference in zip(predictions, references):
        result = scorer.score(
            prep_for_rouge_lsum(reference),
            prep_for_rouge_lsum(prediction)
        )

        for rouge_type in ROUGE_TYPES:
            scores[rouge_type].append(result[rouge_type].fmeasure)

    if use_aggregator:
        return {
            rouge_type: float(np.mean(values))
            for rouge_type, values in scores.items()
        }

    return scores

In [27]:
reference_sentence = "The quick brown fox jumps over the lazy dog."

candidate_sentence = "The quick brown fox jumps over the lazy dog."

scorer = rouge_scorer.RougeScorer(
    ROUGE_TYPES,
    use_stemmer=True
)

scorer.score(reference_sentence, candidate_sentence)

{'rouge1': Score(precision=1.0, recall=1.0, fmeasure=1.0),
 'rouge2': Score(precision=1.0, recall=1.0, fmeasure=1.0),
 'rougeL': Score(precision=1.0, recall=1.0, fmeasure=1.0),
 'rougeLsum': Score(precision=1.0, recall=1.0, fmeasure=1.0)}

In [29]:
candidate_sentence =  "A fast brown fox jumps over a lazy dog."

scorer = rouge_scorer.RougeScorer(
    ROUGE_TYPES,
    use_stemmer=True
)

scorer.score(reference_sentence, candidate_sentence)

{'rouge1': Score(precision=0.6666666666666666, recall=0.6666666666666666, fmeasure=0.6666666666666666),
 'rouge2': Score(precision=0.5, recall=0.5, fmeasure=0.5),
 'rougeL': Score(precision=0.6666666666666666, recall=0.6666666666666666, fmeasure=0.6666666666666666),
 'rougeLsum': Score(precision=0.6666666666666666, recall=0.6666666666666666, fmeasure=0.6666666666666666)}

In [30]:
candidate_sentence =  "The company reported strong quarterly earnings after market close."

scorer = rouge_scorer.RougeScorer(
    ROUGE_TYPES,
    use_stemmer=True
)

scorer.score(reference_sentence, candidate_sentence)

{'rouge1': Score(precision=0.1111111111111111, recall=0.1111111111111111, fmeasure=0.1111111111111111),
 'rouge2': Score(precision=0.0, recall=0.0, fmeasure=0.0),
 'rougeL': Score(precision=0.1111111111111111, recall=0.1111111111111111, fmeasure=0.1111111111111111),
 'rougeLsum': Score(precision=0.1111111111111111, recall=0.1111111111111111, fmeasure=0.1111111111111111)}

In [32]:
print(summary_txt)

print(ref_txt)

researchers at antwerp university in belgium, belgium, permeated a bookshop with the perfume of chocolate . they found customers were 3.5 times more likely to buy romance novels when they smelt chocolate . the government funded study also found the scent encouraged customers to browse through titles .
belgium scientists flooded a bookshop with a chocolatey smell to find customers were almost six times more likely to buy a romance novel . the government-funded study said the scent does not have a big effect on making people want to purchase travel, crime, business or comic books . researchers at antwerp university said the sweet smell encouraged customers to browse and . sales of all books rose during the experiment .


In [31]:
summary_result = compute_rouge(
    predictions=[summary_txt],
    references=[ref_txt],
)

summary_result

{'rouge1': 0.5043478260869565,
 'rouge2': 0.3185840707964602,
 'rougeL': 0.4,
 'rougeLsum': 0.5043478260869565}

In [33]:
articles_txt = cleaned_df['Content']

articles_txt

,Content
0,it seems to be customary for manchester united...
1,during cnn going green: green light for busine...
2,london (cnn) -- police are to investigate clai...
3,(cnn) -- four people were killed and one serio...
4,family believes kristi and benjamin strack kil...
...,...
4995,"by . dan bloom . published: . 08:46 est, 14 no..."
4996,staff at the children ward at southampton gene...
4997,"by . mia de graaf . published: . 14:38 est, 12..."
4998,stores of timber were destroyed in the fire at...


Use `tqdm` to show a progress bar while we summarize the first 50 articles and
collect the candidate summaries.

In [34]:
from tqdm import tqdm

candidate_summaries = []

for i, text in enumerate(tqdm(articles_txt[:50])):
    candidate = summarizer(text)
    candidate_summaries.append(candidate[0]["summary_text"])

100%|██████████| 50/50 [00:45<00:00,  1.10it/s]


In [35]:
summaries_txt = cleaned_df['Summary']

summaries_txt

,Summary
0,victor valdes posed with a fan outside san car...
1,jet republic has teamed up with climatecare to...
2,new: the uk government stands firmly against t...
3,an explosion occurrs at a storage tank that wa...
4,– police suspect foul play and poisoning in th...
...,...
4995,"olivia adams, 13, was told she had broken a st..."
4996,a thief stole a games console from a hospital ...
4997,nitzan benhaim tackled 90ft-tall waves in naza...
4998,a luxury gazebo firm escaped major damage by t...


In [37]:
result_unagg = compute_rouge(
    predictions=candidate_summaries,
    references=list(summaries_txt[:50]),
    use_aggregator=False,
)

result_unagg

{'rouge1': [0.3829787234042554,
  0.41509433962264153,
  0.5084745762711864,
  0.5476190476190477,
  0.27199999999999996,
  0.3779527559055118,
  0.2857142857142857,
  0.5420560747663552,
  0.5043478260869565,
  0.38636363636363635,
  0.36893203883495146,
  0.5168539325842696,
  0.4489795918367347,
  0.09677419354838708,
  0.3188405797101449,
  0.3695652173913043,
  0.1818181818181818,
  0.3783783783783784,
  0.7580645161290323,
  0.2086330935251799,
  0.36111111111111105,
  0.22222222222222224,
  0.273972602739726,
  0.20588235294117646,
  0.42696629213483145,
  0.3333333333333333,
  0.22222222222222224,
  0.3373493975903615,
  0.1276595744680851,
  0.3576158940397351,
  0.42276422764227645,
  0.5794392523364486,
  0.18823529411764706,
  0.48888888888888893,
  0.35999999999999993,
  0.2527075812274368,
  0.1739130434782609,
  0.3958333333333333,
  0.25,
  0.1518987341772152,
  0.5192307692307692,
  0.20338983050847456,
  0.628099173553719,
  0.5614035087719298,
  0.7714285714285715,
 

In [38]:
result_agg = compute_rouge(
    predictions=candidate_summaries,
    references=list(summaries_txt[:50])
)

result_agg

{'rouge1': 0.37251523846904155,
 'rouge2': 0.17544055191068955,
 'rougeL': 0.276947109582718,
 'rougeLsum': 0.34320442199812107}

Find the best and worst summaries by ROUGE-Lsum (min and max indices).

In [39]:
result_unagg_rsum = np.array(result_unagg["rougeLsum"])

max_arg = np.argmax(result_unagg_rsum)
min_arg = np.argmin(result_unagg_rsum)

max_arg, min_arg

(np.int64(44), np.int64(16))

**Worst** candidate summary and its reference:

In [40]:
candidate_summaries[min_arg]

'Manxman kneen, 28, has been ruled out after breaking his arm in a mountain biking crash . mar-train boss tim martin revealed that kneen had undergone surgery but remains hopeful that the manx rider will be fit for next month isle of man tt .'

In [41]:
summaries_txt[min_arg]

'former motorgp star jeremy mcwilliams will replace injured dan kneen in the mar-train yamaha team at this week north west 200.'

**Best** candidate summary and its reference:

In [42]:
candidate_summaries[max_arg]

'janelle duncan-bailey, 25, went missing in south london in the early hours of wed Wednesday . Her body was found yesterday afternoon in mayfield crescent, thornton heath . Jerome mcdonald, 30, has been charged with her murder and will appear in court tomorrow .'

In [43]:
summaries_txt[max_arg]

'janelle duncan-bailey went missing in south london early on wednesday . her body was found yesterday in thornton heath . jerome mcdonald, 30, has been charged with her murder .'

In [44]:
act_vs_pred_summaries_df = pd.DataFrame(
    list(zip(candidate_summaries, summaries_txt[:50])),
    columns=["Predicted_Summaries", "Reference_summaries"],
)

act_vs_pred_summaries_df.head(10)

,Predicted_Summaries,Reference_summaries
0,victor valdes enjoyed a meal out at san carlo ...,victor valdes posed with a fan outside san car...
1,Aviation industry is often seen as one of the ...,jet republic has teamed up with climatecare to...
2,London police to investigate claims that briti...,new: the uk government stands firmly against t...
3,The blast occurred at a storage tank that was ...,an explosion occurrs at a storage tank that wa...
4,Family believes kristi and benjamin strack kil...,– police suspect foul play and poisoning in th...
5,An executive at apple said that the company ha...,apple says the company has no obligation to he...
6,Cell phone novels are written entirely on hand...,hugely popular cell phone novels have created ...
7,The facebook group plus size modeling shared t...,the facebook group plus size modeling . shared...
8,"Researchers at antwerp university in belgium, ...",belgium scientists flooded a bookshop with a c...
9,NEW: egypt prime minister appeals for calm and...,new: three arrested american students identifi...
